# 08 — RMSNorm and gated feature transformations

Modern decoder variants change specific operations, not the overall next-token contract. This notebook isolates two replacements before combining them.

RMSNorm: $\mathrm{RMSNorm}(x)=\gamma\odot x/\sqrt{\mathrm{mean}(x^2)+\epsilon}$, without mean subtraction.
SwiGLU: $\mathrm{down}(\mathrm{SiLU}(\mathrm{gate}(x))\odot\mathrm{up}(x))$.
The gate is not a probability distribution: SiLU can be negative and is not bounded above by one.

## How to work through this notebook

Run setup once. At each checkpoint, write a prediction and try the small implementation before reading its adjacent reference solution. All reference cells run unchanged from top to bottom; exercise cells contain safe, optional starting points. Numerical checks use CPU float64 unless explicitly noted. Agent-verified reference execution is separate from your learning progress.

In [ ]:
from pathlib import Path
import sys, copy, math, inspect
from dataclasses import replace
import torch
from torch import nn
from torch.nn import functional as F
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/dongxi_llms/decoder_lab.py").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the Dongxi_LLMs repository")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from dongxi_llms.decoder_lab import (
    DecoderConfig, TinyDecoder, DecoderBlock, MultiHeadAttention, MLP, RMSNorm,
    layer_norm, rms_norm, rope, attend, parameter_count, analytical_parameters,
    cost_estimate, teaching_batch, next_token_loss, fit_one_batch)
torch.set_num_threads(1)
torch.manual_seed(505)
DTYPE = torch.float64
def close(actual, expected, atol=1e-10, rtol=1e-8):
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
print("CPU reference environment:", torch.__version__)


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
from dongxi_llms import decoder_visuals as viz
def show_visual(figure):
    display(figure)
    plt.close(figure)


## Architecture map — your location in the model

The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The modern route uses RoPE inside attention rather than adding learned absolute positions.

![Architecture map — your location in the model. The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The modern route uses RoPE inside attention rather than adding learned absolute positions.](../figures/chapter-05/day-06-01_rmsnorm_and_swiglu-architecture-map.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

In [ ]:
from dongxi_llms import decoder_architecture as architecture
show_visual(architecture.model_map(focus='modern', modern=True))

## Locate modern replacements in the familiar block

RMSNorm replaces LayerNorm and a SwiGLU MLP replaces the GELU MLP, preserving the residual-width interface. The notebook tests these replacements separately before combining them.

![Locate modern replacements in the familiar block. RMSNorm replaces LayerNorm and a SwiGLU MLP replaces the GELU MLP, preserving the residual-width interface. The notebook tests these replacements separately before combining them.](../figures/chapter-05/day-06-01_rmsnorm_and_swiglu-architecture-detail.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

In [ ]:
show_visual(architecture.block_detail(focus='modern', modern=True))

## Open the gated feature transform

The gate branch applies SiLU before elementwise multiplication with the content branch. The down projection returns to model width. A gate is not an attention probability.

![Open the gated feature transform. The gate branch applies SiLU before elementwise multiplication with the content branch. The down projection returns to model width. A gate is not an attention probability.](../figures/chapter-05/day-06-01_rmsnorm_and_swiglu-architecture-gated-mlp.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

In [ ]:
show_visual(architecture.mlp_detail(gated=True))

## 1. Implement RMSNorm and compare derivatives

What information does mean subtraction remove that RMS scaling does not? Match the explicit formula to a library reference.

**Your prediction:** _Write it here before running the reference._

In [ ]:
x = torch.randn(2, 6, 16, dtype=DTYPE, requires_grad=True)
weight = torch.randn(16, dtype=DTYPE, requires_grad=True)
# Your implementation: output = ...

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
output = x * torch.rsqrt(x.square().mean(-1, keepdim=True)+1e-6) * weight
reference = F.rms_norm(x, (16,), weight, 1e-6)
close(output, reference)
probe = torch.randn_like(x)
for a, b in zip(torch.autograd.grad((output*probe).sum(), (x, weight)),
                torch.autograd.grad((reference*probe).sum(), (x, weight))):
    close(a, b)
ones, zeros = torch.ones(16, dtype=DTYPE), torch.zeros(16, dtype=DTYPE)
close(layer_norm(x, ones, zeros), layer_norm(x+5, ones, zeros))
assert not torch.allclose(rms_norm(x, ones), rms_norm(x+5, ones))
print("Mean before/after RMS scaling:", x.mean(-1).detach(), rms_norm(x, ones).mean(-1).detach())

### Why this works

LayerNorm is invariant to a common feature shift before its affine transform; RMSNorm generally is not. RMS normalization controls root-mean-square magnitude, not the centered variance. Epsilon matters near zero.

### Visual explanation — See the effect of centering

LayerNorm centers the coordinates; RMSNorm retains their common offset while rescaling. This panel is before any learned affine change.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![See the effect of centering. LayerNorm centers the coordinates; RMSNorm retains their common offset while rescaling. This panel is before any learned affine change.](../figures/chapter-05/day-06-01_rmsnorm_and_swiglu-visual-rms-vs-ln.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.features({'Input + 5': (x+5)[0,0], 'LayerNorm': layer_norm(x+5,ones,zeros)[0,0], 'RMSNorm': rms_norm(x+5,ones)[0,0]}, 'LayerNorm and RMSNorm preserve different information'))

## 2. Build the gated MLP explicitly

Inspect both expanded branches, multiply them elementwise, and contract to model width. Test gradients, not just shapes.

**Your prediction:** _Write it here before running the reference._

In [ ]:
gated = MLP(16, 22, gated=True).double()
states = torch.randn(2, 6, 16, dtype=DTYPE, requires_grad=True)
# Your implementation: gate = ...; content = ...; result = ...

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
gate = F.silu(F.linear(states, gated.gate.weight))
content = F.linear(states, gated.up.weight)
manual = F.linear(gate*content, gated.down.weight)
close(manual, gated(states))
probe = torch.randn_like(manual)
a = torch.autograd.grad((manual*probe).sum(), states)[0]
b = torch.autograd.grad((gated(states)*probe).sum(), states)[0]
close(a, b)
print("Gate/content/result shapes:", gate.shape, content.shape, manual.shape)
print("Gate min/max:", float(gate.min().detach()), float(gate.max().detach()))

### Why this works

The two expanded branches see the same state but learn different projections. Their product modulates features at each position; it does not mix tokens.

### Visual explanation — Inspect both branches and their product

The gate modulates content coordinate by coordinate before the down projection. The product can be negative; there is no requirement to sum to one.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![Inspect both branches and their product. The gate modulates content coordinate by coordinate before the down projection. The product can be negative; there is no requirement to sum to one.](../figures/chapter-05/day-06-01_rmsnorm_and_swiglu-visual-gate-product.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.features({'Content': content[0,0], 'SiLU gate': gate[0,0], 'Product': (gate*content)[0,0]}, 'SwiGLU gates features, not token probabilities'))

## 3. Zero the gate and compare parameter budgets

Will a zero gate remove this bias-free MLP's output? Does keeping the same hidden width keep the same parameter count as the GELU MLP?

**Your prediction:** _Write it here before running the reference._

In [ ]:
baseline = MLP(16, 32).double()
same_width = MLP(16, 32, gated=True).double()
# Predict counts and zero-gate behavior first.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
disabled = copy.deepcopy(gated)
with torch.no_grad(): disabled.gate.weight.zero_()
close(disabled(states), torch.zeros_like(states))
print("GELU h=32:", parameter_count(baseline))
print("SwiGLU h=32:", parameter_count(same_width))
print("SwiGLU h=22:", parameter_count(gated))
assert parameter_count(baseline) == 2*16*32 + 32 + 16
assert parameter_count(gated) == 3*16*22
print("Budget difference at h=22:", parameter_count(gated)-parameter_count(baseline))

### Why this works

SwiGLU has three weight matrices rather than two. The common two-thirds width heuristic approximately matches matrix parameters; this lab's GELU biases leave a small residual difference. A zero gate yields zero here because the gated projections have no biases.

## 4. Replace one mechanism at a time

Use identical baseline attention weights while replacing only normalization, only the MLP, and then both. What would a fair quality comparison additionally require?

**Your prediction:** _Write it here before running the reference._

In [ ]:
base = DecoderBlock(DecoderConfig()).double()
# Build ablations from copies, preserving every component not under test.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
norm_only, mlp_only, both = [copy.deepcopy(base) for _ in range(3)]
norm_only.norm1, norm_only.norm2 = RMSNorm(16).double(), RMSNorm(16).double()
new_mlp = copy.deepcopy(gated)
mlp_only.mlp = copy.deepcopy(new_mlp)
both.norm1, both.norm2 = copy.deepcopy(norm_only.norm1), copy.deepcopy(norm_only.norm2)
both.mlp = copy.deepcopy(new_mlp)
reference = base(states)[0]
for name, block in [("norm only", norm_only), ("MLP only", mlp_only), ("both", both)]:
    out = block(states)[0]
    assert torch.isfinite(out).all()
    print(name, "output change norm:", float((out-reference).norm().detach()))

### Why this works

These interventions establish that the replacements change the function while preserving its interface. An untrained output difference is not a quality score. A quality comparison needs declared training/evaluation data and matched budgets.

## Takeaway and evidence boundary

Next: RoPE changes the geometric relationship between queries and keys while keeping token states at model width.

Companion map: [Chapter 5 pathway](../day-05/README.md). Reusable source: [decoder_lab.py](../../src/dongxi_llms/decoder_lab.py). Record your explanation and remaining questions here; the notebook's existence does not mark the lesson complete.